# 🛡️ End-to-End Email Threat Classification
**Domain:** Cybersecurity  
**ML Task:** Binary Classification (Routine vs. Potentially Threatening)  
**Dataset:** `data/spam_dataset.csv` (1,000 email records)  
**Author:** Atanu Das

---
### 🎯 Project Objective
Develop a zero-cost, reproducible machine learning solution that ingests raw email data from `data/spam_dataset.csv`, extracts structural and cybersecurity metadata characteristics, evaluates foundational classification models (Logistic Regression, KNN, Decision Tree, Random Forest), interprets key metrics, and saves model artifacts for deployment.

## 1. Environment Setup & Dependency Imports

In [ ]:
import os
import sys
import re
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.size'] = 11
print("Libraries imported successfully!")

## 2. Dataset Ingestion from `data/spam_dataset.csv`

In [ ]:
# Load local dataset or download via kagglehub fallback
local_csv = os.path.join("data", "spam_dataset.csv")
if os.path.exists(local_csv):
    csv_file = local_csv
    print("Loading local dataset from:", csv_file)
else:
    dataset_path = kagglehub.dataset_download("devildyno/email-spam-or-not-classification")
    csv_file = os.path.join(dataset_path, "spam_dataset.csv")
    print("Downloaded dataset from Kaggle to:", csv_file)

df_raw = pd.read_csv(csv_file)
print(f"Raw Dataset Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
print("--- Missing Values Count ---")
print(df_raw.isnull().sum())
print("\n--- Target Class Balance (is_spam) ---")
print(df_raw['is_spam'].value_counts(normalize=True))

## 3. Real-Time Feature Extraction Engine & Preprocessing
We parse `message_content` to compute 9 structured domain and metadata features required for ML classification:

In [ ]:
FEATURE_COLS = [
    'message_length',
    'url_count',
    'attachment_count',
    'keyword_indicators',
    'domain_age_days',
    'suspicious_tld',
    'spf_dkim_passed',
    'uppercase_ratio',
    'has_external_links'
]

HIGH_RISK_KEYWORDS = [
    'urgent', 'urgently', 'verify', 'verification', 'account', 'bank', 'banking',
    'login', 'password', 'credential', 'credentials', 'suspended', 'suspension',
    'deactivation', 'action required', 'security alert', 'security notice',
    'click here', 'claim', 'prize', 'winner', 'wire transfer', 'update required',
    'update your account', 'unauthorized', 'confirm your', 'access restricted',
    'ssn', 'social security', 'credit card', 'gift card', 'bitcoin', 'crypto'
]

HIGH_RISK_TLDS = ['.xyz', '.top', '.work', '.ru', '.cc', '.info', '.tk', '.click', '.site', '.club', '.online', '.vip']

def extract_features_from_raw_text(text: str) -> dict:
    if not text or not text.strip():
        return None
    clean_text = text.strip()
    message_length = len(clean_text)
    
    urls = re.findall(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', clean_text)
    urls_www = re.findall(r'www\.[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', clean_text)
    url_count = len(set(urls + urls_www))
    
    suspicious_tld = 0
    for tld in HIGH_RISK_TLDS:
        if tld in clean_text.lower():
            suspicious_tld = 1
            break
            
    text_lower = clean_text.lower()
    keyword_indicators = 0
    for kw in HIGH_RISK_KEYWORDS:
        if kw in text_lower:
            keyword_indicators += text_lower.count(kw)
            
    alpha_chars = [c for c in clean_text if c.isalpha()]
    if alpha_chars:
        uppercase_ratio = sum(1 for c in alpha_chars if c.isupper()) / len(alpha_chars)
    else:
        uppercase_ratio = 0.0
        
    attachment_patterns = [r'\.exe', r'\.zip', r'\.rar', r'\.pdf', r'\.doc[x]?', r'attached', r'attachment']
    attachment_count = 0
    for pat in attachment_patterns:
        attachment_count += len(re.findall(pat, text_lower))
        
    if 'spf=pass' in text_lower or 'dkim=pass' in text_lower or 'authentication-results: pass' in text_lower:
        spf_dkim_passed = 1
    elif 'spf=fail' in text_lower or 'dkim=fail' in text_lower or 'spf=softfail' in text_lower:
        spf_dkim_passed = 0
    else:
        spf_dkim_passed = 0 if (suspicious_tld == 1 or keyword_indicators >= 3) else 1
        
    if suspicious_tld == 1 or keyword_indicators >= 4:
        domain_age_days = 45.0
    else:
        domain_age_days = 1200.0
        
    has_external_links = 1 if url_count > 0 else 0
    
    return {
        'message_length': float(message_length),
        'url_count': int(url_count),
        'attachment_count': int(attachment_count),
        'keyword_indicators': int(keyword_indicators),
        'domain_age_days': float(domain_age_days),
        'suspicious_tld': int(suspicious_tld),
        'spf_dkim_passed': int(spf_dkim_passed),
        'uppercase_ratio': float(np.round(uppercase_ratio, 4)),
        'has_external_links': int(has_external_links)
    }

features_list = [extract_features_from_raw_text(t) for t in df_raw['message_content']]
df_processed = pd.DataFrame(features_list)
df_processed['is_threat'] = df_raw['is_spam']

X = df_processed[FEATURE_COLS]
y = df_processed['is_threat']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
print(f"Feature set ({len(FEATURE_COLS)} features): {FEATURE_COLS}")

## 4. Visualizing Feature Relationships & Threat Drivers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(x='is_threat', data=df_processed, ax=axes[0], hue='is_threat', legend=False, palette=['#2ecc71', '#e74c3c'])
axes[0].set_title('Target Class Distribution (0: Routine, 1: Threat)', fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Routine (Ham)', 'Threat (Spam)'])

sns.boxplot(x='is_threat', y='keyword_indicators', data=df_processed, ax=axes[1], hue='is_threat', legend=False, palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('High-Risk Keyword Count by Threat Class', fontweight='bold')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Routine (Ham)', 'Threat (Spam)'])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
corr = df_processed.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix', fontweight='bold')
plt.show()

## 5. Foundational Model Training & Hyperparameter Tuning
We evaluate four classification models using Stratified 5-Fold Cross Validation:
1. **Logistic Regression** (Linear baseline)
2. **K-Nearest Neighbors (KNN)** (Instance-based distance model)
3. **Decision Tree Classifier** (Tree-based non-linear model)
4. **Random Forest** (Ensemble benchmark reference)

In [ ]:
models = {
    'Logistic Regression': (LogisticRegression(random_state=42, max_iter=1000), {'C': [0.1, 1.0, 10.0]}, True),
    'K-Nearest Neighbors': (KNeighborsClassifier(), {'n_neighbors': [3, 5, 7, 9]}, True),
    'Decision Tree': (DecisionTreeClassifier(random_state=42), {'max_depth': [3, 5, 10, None]}, False),
    'Random Forest (Benchmark)': (RandomForestClassifier(random_state=42), {'n_estimators': [50, 100], 'max_depth': [5, 10]}, False)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
eval_results = {}
y_preds = {}
y_probs = {}
best_estimators = {}

for name, (clf, param_grid, use_scaled) in models.items():
    X_tr = X_train_s if use_scaled else X_train
    X_te = X_test_s if use_scaled else X_test
    
    grid = GridSearchCV(clf, param_grid, cv=cv, scoring='f1', n_jobs=-1)
    grid.fit(X_tr, y_train)
    best = grid.best_estimator_
    best_estimators[name] = best
    
    pred = best.predict(X_te)
    prob = best.predict_proba(X_te)[:, 1]
    
    y_preds[name] = pred
    y_probs[name] = prob
    
    eval_results[name] = {
        'Best Params': grid.best_params_,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1-Score': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, prob)
    }

results_df = pd.DataFrame(eval_results).T
results_df

## 6. Detailed Model Evaluation & Metric Interpretation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.flatten()
for idx, (name, pred) in enumerate(y_preds.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Routine (0)', 'Threat (1)'],
                yticklabels=['Routine (0)', 'Threat (1)'])
    axes[idx].set_title(f"{name}\nF1: {eval_results[name]['F1-Score']:.3f}", fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
for name, prob in y_probs.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {eval_results[name]['ROC-AUC']:.3f})", linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.7)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves Comparison', fontweight='bold')
plt.legend(loc='lower right')
plt.show()

## 7. Model Serialization & Streamlit Deployment Artifacts
We save the trained Logistic Regression model and StandardScaler to disk for use in `app.py`.

In [ ]:
os.makedirs('model', exist_ok=True)

with open('model/model.pkl', 'wb') as f:
    pickle.dump(best_estimators['Logistic Regression'], f)

with open('model/vectorizer.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model artifacts (model.pkl, vectorizer.pkl) successfully serialized to model/ directory!")